In [1]:
import sqlite3

from tqdm import tqdm
import pandas as pd
import numpy as np
import chromadb
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity

### Initialize the embedding model

In [2]:
embed_model = OpenAI(base_url="http://localhost:12434/engines/llama.cpp/v1", api_key="not-needed")

In [3]:
# 1. Define the path to your SQLite database file
# Make sure 'Submission.sqlite' is in the same directory as your Python script,
# or provide the full path to the file.
db_file = 'assignment/Submission.sqlite'

# 2. Connect to the SQLite database
# This creates a connection object. If the database file doesn't exist,
# it will be created (though for this dataset, it should already exist).
conn = sqlite3.connect(db_file)
print(f"Successfully connected to {db_file}")

# 3. Create a cursor object
# A cursor allows you to execute SQL commands.
cursor = conn.cursor()

# 4. Define the columns you want to select from the 'exercises_result' table
# Based on the dataset description, available columns include:
# submission_id, submitted_answer, submission_time, exercise_id,
# is_correct, student_id, category [1]
columns_to_select = [
    "preamble",
    "submitted_answer",
    "category",
    "difficulty",
    "ref"
]

# --- 2. Read all tables ---
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
table_names = tables["name"].tolist()
table_names.remove("exercises_benchmark")
table_names.remove("exercises_result")
table_names.remove("exercises_exercise")
print(f"\nExecuting query: {table_names}")

Successfully connected to assignment/Submission.sqlite

Executing query: ['Award', 'restriction', 'Restriction_Category', 'Scene', 'Writer_Award', 'Director_Award', 'movie', 'Actor_Award', 'Appearance', 'Crew', 'Crew_Award', 'movie_award', 'Role', 'person', 'director', 'writer', 'Validation']


### Connect to ChromaDB server

In [4]:
# Example setup of the client to connect to your chroma server
client = chromadb.HttpClient(host='192.168.1.2', port=8033)

client.list_collections()

[Collection(name=moviedb_sql_exercises)]

### Initialize the collection for SQL assignment

In [5]:
try:
    try:
        client.delete_collection(name="moviedb_sql_exercises")
    except Exception as e:
        print("No existing collection to delete.")

    collection = client.create_collection(name="moviedb_sql_exercises", configuration={"hnsw": {"space": "cosine"}})

    # store anoteted assignments
    annotated_assignments = [
        """Question: Assume persons who were born in the same year are the same age and there is only one youngest person (with no ties/draws) in this database, who is/are the second youngest person(s) in the database? List the id(s) of the person(s). Correct: SELECT p.id FROM person p WHERE p.year_born = (SELECT MAX(year_born) FROM person WHERE year_born < (SELECT MAX(year_born) FROM person));
            - Answer: DELETE FROM person p1 WHERE year_born=(SELECT MAX(year_born) FROM person p1）SELECT id FROM person p2 WHERE year_born=(SELECT MAX(year_of_born)FROM person p2); The student firstly deletes the youngest person from the database, then selects the id of the youngest person from the remaining persons. This approach is not allowed as it modifies the database state and does not reflect a valid SQL query for the problem. Student grade: 0, category: cheating
            - Answer: select id from person where year_born in (select year_born from person order by year_born desc limit 1 offset 1); The solution is slightly different but still correct. Student grade: 93, category: correct.
            - Answer: select * from movie_award m, crew_award c, director_award d where m.title; The student's query is syntactically correct but does not make any sense in the context of the question. Student grade: 2, category: partially correct.""",
        """Question: Assume persons who were born in the same year are the same age and there is only one youngest person (with no ties/draws) in this database, who is/are the second youngest person(s) in the database? List the id(s) of the person(s). Correct: SELECT p.id FROM person p WHERE p.year_born = (SELECT MAX(year_born) FROM person WHERE year_born < (SELECT MAX(year_born) FROM person));
            - Answer: select id from person where year_born=1988; The student is directly using the knowledge that the second youngest person was born in 1988, which bypasses the core learning objective of the problem. Student category: cheating.
            - Answer: select id from ((select * from person) except (select * from person where age=max(year_born))as A where age=max(A.year_born); Here the student used 'age=max(year_born)', which is syntactically incorrect and makes the query non-interpretable. Student grade: 0, category: noninterpretable.
            - Answer: SELECT id FROM person WHERE id != '00000903' ORDER BY year_born DESC LIMIT 1; The student directly uses the knowledge that the youngest person has id '00000903', which bypasses the core learning objective of the problem. Student grade: 68, category: cheating.""",
        """Question: Which movies were written by Kevin Williamson? List the titles and production years of these movies. Correct: SELECT w.title, w.production_year FROM writer w, person p WHERE w.id = p.id AND lower(first_name)= 'kevin' AND lower(last_name)= 'williamson';
            - Answer: select title, production_year from writer where id='00000402'; The student directly uses the knowledge that Kevin Williamson has id '00000402', which bypasses the core learning objective of the problem. Student grade: 63, category: cheating.
            - Answer: SELECT title, production_year FROM person NATURAL JOIN movie NATURAL JOIN writer GROUP BY title, production_year HAVING first_name="Kevin" AND last_name="Williamson"; The student correctly answer the question, but they did not lowercase the first_name and last_name, which makes the query less robust. Student category: correct.
            - Answer: SELECT m.title,m.production_year FROM movie m WHERE EXISTS(SELECT w.title FROM (person p NATURAL JOIN writer) AS w WHERE w.first_name = 'Kevin' AND w.last_name = 'Williamson'); The solution does not normalize case (e.g., via LOWER()), so it’s sensitive to capitalization differences in names. Student category: partially correct.""",
        """Question: Which movies were written by Kevin Williamson? List the titles and production years of these movies. Correct: SELECT w.title, w.production_year FROM writer w, person p WHERE w.id = p.id AND lower(first_name)= 'kevin' AND lower(last_name)= 'williamson';
            - Answer: SELECT TITLE, PRODUCTION_YEAR FROM WRITER W WHERE TITLE IN(SELECT TITLE FROM WRITER WHERE ID IN(SELECT ID FROM PERSON WHERE FIRST_NAME = KEVIN AND LAST_NAME = WILLIAMSON)) GROUP BY TITLE; The student has omitted the apostrophes around the string literals 'KEVIN' and 'WILLIAMSON', which makes the query non-interpretable. Student category: noninterpretable.
            - Answer: select pw.title, pw.production_year from (person natural join writer) as pw where lower(pw.first_name)='kevin' and lower(pw.last_name)='williamson'; The solution is slightly different but still correct. Student category: correct.""",
        """Question: How many writers were born in 1935? Correct: SELECT count(*) FROM person p WHERE EXISTS (SELECT * FROM writer w WHERE w.id = p.id AND p.year_born = 1935);
            - Answer: SELECT COUNT(DISTINCT p.id) FROM person p INNER JOIN writer w ON p.id = w.id WHERE p.year_born = 1935; The solution is slightly different but still correct. Student category: correct.
            - Answer: select count(distinct 1935)from writer; Here in the database is presented only one writer born in 1935, so the student answer produces the correct execution result, but the method used blatantly bypasses the core learning objective of the problem. Student grade: 0, category: cheating.""",
        """Question: How many writers were born in 1935? Correct: SELECT count(*) FROM person p WHERE EXISTS (SELECT * FROM writer w WHERE w.id = p.id AND p.year_born = 1935);
            - Answer: select x.count_number from (select count(*)as count_number,p.id from writer as w, person as p where p.year_born ='1935' and w.id=p.id)as x; The student incorrectly calculated the number of writers because they counted each title a person wrote, rather than counting each unique person, which overestimates the total number of writers. Student category: partially correct.
            - Answer: select count(*) from Writer w inner join Person p on w.id = p.id where year_born = 1935; The student incorrectly calculated the number of writers because they counted each title a person wrote, rather than counting each unique person, which overestimates the total number of writers. Student category: partially correct. Student category: partially correct.""",
        """Question: Who have directed at least two movies that were written by themselves (i.e., a director is one of the writers for the same movie)? Show their ids, the ﬁrst and last names. Correct: SELECT dw.id, dw.first_name, dw.last_name FROM (person p NATURAL JOIN director NATURAL JOIN writer) AS dw GROUP BY dw.id HAVING count(*) > 1;
            - Answer: Select d.id, p.first_name, p.last_name From Director d, Person p Where Exists(Select * From Writer w Where d.title = w.title AND p.id = w.id) Group By Having Count(*) > 1; The student's query is syntactically incorrect due to the missing column name after 'Group By'. Student category: noninterpretable.
            - Answer: select * from actor_award; The student's query is syntactically correct, but it is logically wrong in the context of the question. Student category: partially correct.""",
        """Question: Who have directed at least two movies that were written by themselves (i.e., a director is one of the writers for the same movie)? Show their ids, the ﬁrst and last names. Correct: SELECT dw.id, dw.first_name, dw.last_name FROM (person p NATURAL JOIN director NATURAL JOIN writer) AS dw GROUP BY dw.id HAVING count(*) > 1;
            - Answer: with writer2 as (select id from writer group by id having count(id)>1), director2 as (select id from director group by id having count(id)>1), l as (select writer2.id from director2, writer2 where director2.id = writer2.id) select person.id, first_name, last_name from person, l  where person.id = l.id; The student's query is syntactically correct but more complex than necessary. Student category: correct.
            - Answer: SELECT dw.id, dw.first_name, dw.last_name FROM (person p NATURAL JOIN director NATURAL JOIN writer) AS dw GROUP BY dw.id HAVING count(*) > 1; The solution is exactly the same as the reference solution. Student grade: 100, category: correct.""",
        """Question: Who have directed at least two movies that were written by themselves (i.e., a director is one of the writers for the same movie)? Show their ids, the ﬁrst and last names. Correct: SELECT dw.id, dw.first_name, dw.last_name FROM (person p NATURAL JOIN director NATURAL JOIN writer) AS dw GROUP BY dw.id HAVING count(*) > 1;
            - Answer: select person.id,first_name,last_name from(select id from director d where exists (select * from writer w where w.id = d.id and w.title = d.title and w.production_year = d.production_year)) as mm,person where mm.id = person.id group by person.id having count(person.id)>1; The solution is slightly different but still correct. Student category: correct.
            - Answer: SELECT DISTINCT id, first_name, last_name FROM (person NATURAL JOIN writer NATURAL JOIN director) AS dw GROUP BY dw.id HAVING COUNT(dw.id)>1; The solution is slightly different but still correct. Student category: correct.
            - Answer: select person.id,person.first_name,person.last_name from person join director on director.id = person.id join writer on writer.id = person.id and director.title = writer.title group by person.id having count(*) >1; The student has slightly different solution but still correct. Student category: correct.""",
        """Question: Who have been nominated for a director award at least once, but have never won any director award? List their ids, and the award names, years of awards and categories which they have been nominated for. Correct: SELECT d.id, da.award_name, da.year_of_award, da.category FROM director_award da, director d WHERE da.title = d.title AND da.production_year = d.production_year AND lower(da.result)= 'nominated' AND d.id NOT IN ( SELECT id FROM director_award da, director d WHERE da.title = d.title AND da.production_year = d.production_year AND lower(da.result)= 'won' );
            - Answer: select da.id,da.award_name,da.year_of_award, da.category from (director_award natural join director) da where da.result = 'nominated' and da.id not in ( select id from (director_award natural join director) da where da.result = 'won') ; The solution is slightly different but still correct. Student category: correct.
            - Answer: WITH never_won AS (SELECT * FROM director_award INNER JOIN director USING(title,production_year) WHERE result = 'nominated') SELECT id, award_name, year_of_award, category FROM never_won; The solution is slightly different but still correct. Student category: correct.
            - Answer: select d.id,ma.award_name,ma.year_of_award,ma.category from director d, director_award ma where lower(ma.result)='nominated' and lower(ma.result)!='won' and ma.title=d.title and ma.production_year=d.production_year; The solution is slightly different but still correct. Student category: correct.""",
        """Question: How many writer awards have been given to Woody Allen between 1991 and 1995 (inclusive)? List the number of the awards. Correct: SELECT COUNT(*) AS number_of_award FROM writer_award wa, writer w, person p WHERE wa.title = w.title AND wa.production_year = w.production_year AND wa.id = w.id AND w.id = p.id AND lower(RESULT)= 'won' AND lower(first_name)= 'woody' AND lower(last_name)= 'allen' AND year_of_award >= 1991 AND year_of_award <= 1995;
            - Answer: select count(*) from crew, crew_award where crew.id = crew_award.id and lower(crew_awrad.result) = 'won'; The student's query is syntactically incorrect due to the typo in 'crew_awrad', which should be 'crew_award', making it non-interpretable. Student category: noninterpretable.
            - Answer: SELECT COUNT(id) as NumberOfAwards FROM (PERSON NATURAL JOIN WRITER_AWARD) as pa WHERE lower(pa.first_name) = "woody" and lower(pa.last_name) = "allen" and pa.year_of_award >= 1991 and pa.year_of_award <= 1995; The solution can be executed, it is incorrect because it does not include condition to filter for awards that were actually 'won' Student category: partially correct.
            - Answer: select count(*) from writer_award wa, person p, director d where p.id = d.id and lower(p.first_name) = 'woody' and lower(p.last_name) = 'allen' and wa.year_of_award >= 1991 and wa.year_of_award <= 1995; The solution can be executed, it is incorrect because it does not include the necessary conditions to filter for awards that were actually 'won', nor the 'title' or 'production_year' from the relevant tables. Student category: partially correct.""",        
        """Question: How many writer awards have been given to Woody Allen between 1991 and 1995 (inclusive)? List the number of the awards. Correct: SELECT COUNT(*) AS number_of_award FROM writer_award wa, writer w, person p WHERE wa.title = w.title AND wa.production_year = w.production_year AND wa.id = w.id AND w.id = p.id AND lower(RESULT)= 'won' AND lower(first_name)= 'woody' AND lower(last_name)= 'allen' AND year_of_award >= 1991 AND year_of_award <= 1995;
            - Answer: select count(*) from writer_award where year_of_award >1991 and year_of_award <=1995 and id=(select id from person where first_name='Woody' and last_name='Allen'); The solution is exactly the same as the reference solution. Student category: correct.
            - Answer: select count(*) from writer_award wa, writer w,person p where w.id=p.id and lower(p.first_name)='woody' and lower(p.last_name)='allen' and w.title=wa.title and w.production_year=wa.production_year and wa.year_of_award>=1991 and wa.year_of_award<=1995 and lower(wa.result)='won'; The solution is exactly the same as the reference solution. Student grade: 100, category: correct.""",        
        """Question: How many movies were produced in 1993, 1992 and 1991? List the production years and the corresponding numbers of movies. Correct: SELECT production_year, count(*) FROM movie WHERE production_year > 1990 AND production_year < 1994 GROUP BY production_year;
            - Answer: SELECT production_year, count(*) FROM movie WHERE production_year > 1990 AND production_year < 1994 GROUP BY production_year; The solution is exactly the same as the reference solution. Student grade: 100, category: correct.""",
        """Question: List the titles and production years of all movies that have won a "BAFTA Film Award" under the category ‘Best Film’, together with the ﬁrst names and last names of their directors. Correct: SELECT ma.title, ma.production_year, p.first_name, p.last_name FROM ( SELECT title, production_year FROM movie_award WHERE lower(award_name)= 'bafta film award' AND lower(category)= 'best film' AND lower(RESULT)= 'won' ) AS ma, director AS d, person AS p WHERE ma.title = d.title AND ma.production_year = d.production_year AND d.id = p.id;
            - Answer: SELECT ma.title, ma.production_year, p.first_name, p.last_name FROM person p, MOVIE_AWARD ma WHERE EXISTS (SELECT * FROM DIRECTOR d WHERE d.id=p.id AND d.production_year=ma.production_year AND d.title=ma.title) AND lower(ma.result)='won' AND lower(ma.award_name)='bafta film award' AND lower(ma.category)='best film'; The solution is slightly different but still correct. Student category: correct.
            - Answer: SELECT ma.title, ma.production_year, p.first_name, p.last_name FROM ( SELECT title, production_year FROM movie_award WHERE lower(award_name)= 'bafta film award' AND lower(category)= 'best film' AND lower(RESULT)= 'won' ) AS ma, director AS d, person AS p WHERE ma.title = d.title AND d.id = p.id; The solution is slightly different but still correct. Student category: correct.""",
        """Question: List the titles and production years of all movies that have won a "BAFTA Film Award" under the category ‘Best Film’, together with the ﬁrst names and last names of their directors. Correct: SELECT ma.title, ma.production_year, p.first_name, p.last_name FROM ( SELECT title, production_year FROM movie_award WHERE lower(award_name)= 'bafta film award' AND lower(category)= 'best film' AND lower(RESULT)= 'won' ) AS ma, director AS d, person AS p WHERE ma.title = d.title AND ma.production_year = d.production_year AND d.id = p.id;
            - Answer: select title, production_year,last_name,first_name from movie_award,director where award_name='BAFTA Film Award' and category='best film';  The student's query contains ambiguous column name: title and production_year, which makes it non-interpretable. Student grade: 47, category: noninterpretable.
            - Answer: select Movie_Award.title, Movie_Award.production_year, person.first_name, person.last_name from  Movie_Award join Director on Director.title= Movie_Award.title join person on person.id = director.id where movie_Award.category = 'Best Film' and movie_Award.award_name = 'BAFTA Film Award' and movie_Award.award_name = 'won' group by Movie_Award.title; The student's query uses movie_award.award_name = 'won', but 'won' is not a valid award name in the database, and they failed to lowercase award_name, category, and result. Student category: partially correct.""",
        """Question: How many persons have never directed any movies in this database? List the total number of such persons. Correct: SELECT count(*) FROM person p WHERE NOT EXISTS (SELECT * FROM director d WHERE p.id = d.id);
            - Answer: selcet count(p.id) from person p, director d where p.id != d.id; The student's query contains a typo in the 'selcet', which should be 'select', making it non-interpretable. Student category: noninterpretable.
            - Answer: select id, count(*) from PERSON inner join DIRECTOR on PERSON.id = DIRECTOR.id; The student's query contains ambiguous column name: id which makes it non-interpretable. Student grade: 33, category: noninterpretable.
            - Answer: SELECT count(*) FROM(SELECT DISTINCT id FROM director); The solution gives the number of directors, not the number of persons who have never directed any movies. Student category: partially correct.""",
        """Question: A person has worked on a movie if this person is a director, a writer, or both a director and writer of this movie. Who has/have worked on the largest number of distinct movies in this database? List the id(s) of the person(s). Correct: SELECT x.id FROM (SELECT x1.id, count(*) AS NoOfWorkedON FROM (SELECT d.id, title, production_year FROM director d UNION SELECT w.id, title, production_year FROM writer w) x1 GROUP BY x1.id) x WHERE x.NoOfWorkedON = (SELECT MAX(x2.NoOfWorkedON) FROM (SELECT x1.id, count(*) AS NoOfWorkedON FROM (SELECT d.id, title, production_year FROM director d UNION SELECT w.id, title, production_year FROM writer w) x1 GROUP BY x1.id) x2)
            - Answer: select X.id from (select p.id, count (*) as lnm from person p, (select id from writer w union select id from director d) as x1 where x1.id=p.id group by p.id) as X where x.lnm in (select max(x2.lnm) from (select p.id, count (*) as lnm from person p,(select id from  writer union select id from director ) as x3 where x3.id=p.id group by p.id) as X2); The student miss during union the title, production_year columns, which is crucial for accurately counting distinct movies. Student category: partially correct.
            - Answer: select id from (select id,count(*)as number from(select id,title,production_year from director union select id,title,production_year from writer)as x group by x.id having x.number in (select max(x3.number) from (select id,count(*)as number from(select id,title,production_year from director union select id,title,production_year from writer)as x2 group by x2.id)as x3))as w; The student's query is syntactically incorrect because they are using 'having x.number' in the HAVING clause, which should be only 'having number'. Student category: noninterpretable.""",
        """Question: Who has played the largest number of roles in this database (note that: the roles may be in the same or diﬀerent movies)? List the ﬁrst name, last name and the roles' information (i.e. the title, production year, description) of that person. Correct: SELECT p.first_name, p.last_name, r.title, r.production_year, r.description FROM person p, ROLE r, (SELECT id, count(*) AS number_of_roles FROM ROLE GROUP BY id) AS b WHERE r.id = p.id AND p.id = b.id AND b.number_of_roles IN (SELECT max(number_of_roles) FROM (SELECT id, count ( * ) AS number_of_roles FROM ROLE GROUP BY id) AS a);
            - Answer: select p.first_name, p.last_name, r.title, r.production_year, r.description from person p, role r, (select id, count(*) as no_of_roles from role group by id)b where p.id=r.id and p.id=b.id and b.no_of_roles in (select max (no_of_roles) from ( select id, count(*) as no_of_roles from role group by id) a); The solution is slightly different but still correct. Student category: correct.""",
        """Question: Who has played the largest number of roles in this database (note that: the roles may be in the same or diﬀerent movies)? List the ﬁrst name, last name and the roles' information (i.e. the title, production year, description) of that person. Correct: SELECT p.first_name, p.last_name, r.title, r.production_year, r.description FROM person p, ROLE r, (SELECT id, count(*) AS number_of_roles FROM ROLE GROUP BY id) AS b WHERE r.id = p.id AND p.id = b.id AND b.number_of_roles IN (SELECT max(number_of_roles) FROM (SELECT id, count ( * ) AS number_of_roles FROM ROLE GROUP BY id) AS a);
            - Answer: SELECT person.first_name, person.last_name, role.title, role.production_year, role.description FROM person, role WHERE person.id = role.id GROUP BY person.id HAVING role; The student's query is syntactically incorrect due to the incomplete HAVING clause. Student category: noninterpretable.
            - Answer: select id,first_name,last_name,title,production_year,description,count(*) as c from (role NATURAL JOIN person) as r where r.c=(select count(*) from (role NATURAL JOIN person) as r group by id) group by id; Student category: noninterpretable.""",
        """Question: Who appeared in exactly one scene in the movie ‘Psycho’ produced in 1960? List ids of these actors/actresses. Correct: SELECT r.id FROM scene s, appearance a, ROLE r WHERE s.title = a.title AND s.production_year = a.production_year AND s.scene_no = a.scene_no AND s.title = r.title AND s.production_year = r.production_year AND a.description = r.description AND lower(a.title) = 'psycho' AND a.production_year = 1960 GROUP BY r.id HAVING count(*) = 1;
            - Answer: SELECT r.id FROM scene s, appearance a, ROLE r WHERE s.title = a.title AND s.production_year = a.production_year AND s.scene_no = a.scene_no AND r.title = a.title AND r.production_year = a.production_year AND r.description = a.description AND lower(a.title) = 'psycho' AND a.production_year = 1960 GROUP BY r.id HAVING count(*) = 1; The solution is exactly the same as the reference solution. Student category: correct.
            - Answer: SELECT r.id FROM scene s, role r, appearance a WHERE s.production_year=r.production_year AND s.title=r.title AND s.scene_no=a.scene_no AND r.description=a.description AND r.title='Psycho' AND r.production_year='1960' GROUP BY r.id HAVING COUNT(*)=1; The student's query contains a typo in the column name 'prodction_year', which should be 'production_year', making it non-interpretable. Student category: noninterpretable.""",
        """Question: Who appeared in exactly one scene in the movie ‘Psycho’ produced in 1960? List ids of these actors/actresses. Correct: SELECT r.id FROM scene s, appearance a, ROLE r WHERE s.title = a.title AND s.production_year = a.production_year AND s.scene_no = a.scene_no AND s.title = r.title AND s.production_year = r.production_year AND a.description = r.description AND lower(a.title) = 'psycho' AND a.production_year = 1960 GROUP BY r.id HAVING count(*) = 1;
            - Answer: select r.id from role r, scene s, appearance a where s.title = a.title and s.production_year = a.production_year and s.scene_no = a.scene_no and s.title = r.title and s.production_year = r.production_year and lower(s.title) = 'psycho' and s.production_year = 1960 group by r.id having count(*) = 1; Here the student missed the join condition 'AND a.description = r.description', which is crucial for accurately linking roles to appearances. Student category: partially correct.""",
        """Question: Who appeared in exactly one scene in the movie ‘Psycho’ produced in 1960? List ids of these actors/actresses. Correct: SELECT r.id FROM scene s, appearance a, ROLE r WHERE s.title = a.title AND s.production_year = a.production_year AND s.scene_no = a.scene_no AND s.title = r.title AND s.production_year = r.production_year AND a.description = r.description AND lower(a.title) = 'psycho' AND a.production_year = 1960 GROUP BY r.id HAVING count(*) = 1;
            - Answer: select s.id from scene s where exists (select * from movie m where m.title = s.title ='Psycho' and m.production_year = s.production_year = 1960 and s.scene_no = 1 ); The student's query is non-interpretable because it references an invalid column named 'id' from the 'scene' table. Student category: noninterpretable.
            - Answer: select * from SCENE; The student's query is syntactically correct but does not make any logically sense in the context of the question. Student category: partially correct.""",
        """Question: Who is the youngest person in the database? List the id, ﬁrst name, and last name of this person. Correct: SELECT id, first_name, last_name FROM person p WHERE p.year_born = (SELECT max(year_born) FROM person );
            - Answer: select p.id,p.first_name,p.last_name from person p where p.year_born = (select year_born	from person	order by year_born desc limit 1); The solution is slightly different but still correct. Student grade: 100, category: correct.
            - Answer: select id,first_name,last_name from person group by id having (year_born=1992); The student is directly using the knowledge that the youngest person was born in 1992, which bypasses the core learning objective of the problem. Student category: cheating.""",
        """Question: Who is the youngest person in the database? List the id, ﬁrst name, and last name of this person. Correct: SELECT id, first_name, last_name FROM person p WHERE p.year_born = (SELECT max(year_born) FROM person );
            - Answer: SELECT id, first_name, last_name FROM person p WHERE p.year_born = (SELECT MIN(p1.year_born) FROM person p1); The student is confusing the logic for finding the oldest and youngest person, mistakenly using the minimum birth year to find the youngest person instead of the maximum birth year. Student grade: 47, category: partially correct.
            - Answer: SELECT id,first_name,last_name FROM person p WHERE  MAX(p.year_born); The student's query is syntactically incorrect because the MAX() function cannot be used in the WHERE clause in this manner, making it non-interpretable. Student category: noninterpretable.""",
        """Question: Which countries have restricted the movie ‘Shakespeare in Love’ as ‘M’? List the names of these countries. Correct: SELECT country FROM restriction WHERE lower(title) = 'shakespeare in love' AND description = 'M';
            - Answer: SELECT country FROM movie NATURAL JOIN restriction_category GROUP BY country HAVING lower(title)='Shakespeare in Love' AND restriction_category.description='M'; The student's query incorrectly joins the 'Movie' and 'Restriction_Category' tables on the 'country' column, rather than using the 'Restriction' table directly. This is because the 'country' column in the 'Movie' table represents the filming location, whereas the 'country' column in the 'Restriction' table indicates the viewing country. Student category: partially correct.""",
        """Question: How many movies have never won any award, i.e., received none of movie awards, crew awards, director awards, writer awards and actor awards? List the total number of such movies stored in the database. Correct: SELECT count(*) FROM movie m WHERE not exists ( SELECT title, production_year FROM movie_award ma WHERE lower(ma.result) = 'won' and ma.title = m.title and ma.production_year = m.production_year UNION SELECT title, production_year FROM crew_award ca WHERE lower(ca.result) = 'won' and ca.title = m.title and ca.production_year = m.production_year UNION SELECT title, production_year FROM director_award da WHERE lower(da.result) = 'won' and da.title = m.title and da.production_year = m.production_year UNION SELECT title, production_year FROM actor_award aa WHERE lower(aa.result) = 'won' and aa.title = m.title and aa.production_year = m.production_year UNION SELECT title, production_year FROM writer_award wa WHERE lower(wa.result) = 'won' and wa.title = m.title and wa.production_year = m.production_year );
            - Answer: select * from movie; The student's query is syntactically correct but does not make any logically sense in the context of the question. Student category: partially correct."""
    ]
    for document in tqdm(annotated_assignments):
        # Compute embeddings for the batch (one embedding per document)
        embedding = embed_model.embeddings.create(
            model="ai/embeddinggemma:300M-Q8_0",
            input=document,
        ).data[0].embedding


        # Single add() call per batch
        collection.add(
            ids=f"annotated_assignment_{annotated_assignments.index(document)}",
            embeddings=embedding,
            documents=document,
        )
    print("Annotated assignments stored in the collection.")

    # Store each table in the database
    batch_size = 8
    for table in tqdm(table_names):
        # Read the table into a DataFrame
        # df = pd.read_sql(f"SELECT * FROM {table}", conn)
        df = pd.read_sql(f"SELECT * FROM {table}", conn)
        # print(df.head())

        # Iterate over the DataFrame in chunks of 8: [0:8], [8:16], [16:24], ...
        chunk_id = 0
        for start in range(0, len(df), batch_size):
            end = min(start + batch_size, len(df))
            batch = df.iloc[start:end]

            # Build documents for this batch
            document = ""
            for _, row in batch.iterrows():
                for col in batch.columns:
                    document += f"{col}: {str(row[col])} | "
                document += "\n\n"
            # print(document)

            # Compute embeddings for the batch (one embedding per document)
            embedding = embed_model.embeddings.create(
                model="ai/embeddinggemma:300M-Q8_0",
                input=document,
            ).data[0].embedding

            # Single add() call per batch
            collection.add(
                ids=f"table_{table}_chunk_{chunk_id}",
                embeddings=embedding,
                documents=document,
            )

            # print(document, f"assignment_{category}_chunk_{chunk_id}")
            print(f"Added rows {start}-{end - 1} (table size: {len(df)})")
            chunk_id += 1

        print(f"Table {table} from `moviedb` database stored in the collection.")
except Exception as e:
    print(f"{e}")

100%|██████████████████████████████████████████████████████████████████████████████| 26/26 [01:08<00:00,  2.65s/it]


Annotated assignments stored in the collection.


  0%|                                                                                       | 0/17 [00:00<?, ?it/s]

Added rows 0-7 (table size: 10)


  6%|████▋                                                                          | 1/17 [00:03<01:02,  3.92s/it]

...

Added rows 96-101 (table size: 102)
Table director from `moviedb` database stored in the collection.
Added rows 0-7 (table size: 111)
Added rows 8-15 (table size: 111)
Added rows 16-23 (table size: 111)
Added rows 24-31 (table size: 111)
Added rows 32-39 (table size: 111)
Added rows 40-47 (table size: 111)
Added rows 48-55 (table size: 111)
Added rows 56-63 (table size: 111)
Added rows 64-71 (table size: 111)
Added rows 72-79 (table size: 111)
Added rows 80-87 (table size: 111)
Added rows 88-95 (table size: 111)
Added rows 96-103 (table size: 111)


 94%|█████████████████████████████████████████████████████████████████████████▍    | 16/17 [16:32<01:03, 63.77s/it]

Added rows 104-110 (table size: 111)
Table writer from `moviedb` database stored in the collection.


100%|██████████████████████████████████████████████████████████████████████████████| 17/17 [16:33<00:00, 58.47s/it]

Added rows 0-0 (table size: 1)
Table Validation from `moviedb` database stored in the collection.


In [6]:
# It's good practice to close the connection when you're done.
if conn:
    conn.close()
    print(f"\nConnection to {db_file} closed.")


Connection to assignment/Submission.sqlite closed.


In [7]:
# an example input
input = "Represent this query for searching relevant code: Your task is to answer the following questions using SQL queries. Use a single SQL query that may contain subqueries. Question: How many writers were born in 1935? Student's answer: select count(distinct 1935)from writer;"

# generate an embedding for the input and retrieve the most relevant doc
embedding = embed_model.embeddings.create(
    model="ai/embeddinggemma:300M-Q8_0",
    input=("Represent this query for searching relevant code: " + input),
).data[0].embedding
# print(embedding)
results = collection.query(
  query_embeddings=[embedding],
  n_results=10,
)
for i in range(min(len(results['documents'][0]), 10)):
    print(results['documents'][0][i], '\n')


Question: How many writers were born in 1935? Correct: SELECT count(*) FROM person p WHERE EXISTS (SELECT * FROM writer w WHERE w.id = p.id AND p.year_born = 1935);
            - Answer: select x.count_number from (select count(*)as count_number,p.id from writer as w, person as p where p.year_born ='1935' and w.id=p.id)as x; The student incorrectly calculated the number of writers because they counted each title a person wrote, rather than counting each unique person, which overestimates the total number of writers. Student category: partially correct.
            - Answer: select count(*) from Writer w inner join Person p on w.id = p.id where year_born = 1935; The student incorrectly calculated the number of writers because they counted each title a person wrote, rather than counting each unique person, which overestimates the total number of writers. Student category: partially correct. Student category: partially correct. 

Question: How many writers were born in 1935? Correct: SELE